In [ ]:
import numpy as np
import cv2
import os
import json
import torch
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import pickle
from pycocotools.coco import COCO
import skimage.io as io
import albumentations as A
import segmentation_models_pytorch as smp
from monai.losses import DiceCELoss
from torch.cuda.amp import autocast, GradScaler
import torch.nn.functional as F
from torchmetrics.segmentation import DiceScore, MeanIoU
from matplotlib.colors import ListedColormap
import matplotlib.pyplot as plt

In [ ]:
annTrain = './annotations/instances_train2017.json'
valTrain = './annotations/instances_val2017.json'

cocoTrain = COCO(annTrain)
cocoVal = COCO(valTrain)

class_labels = {
    'car': 1,
    'bus': 2,
    'motorcycle': 3,
    'truck': 4,
    'person': 5
}

In [ ]:
trainingImagePath = './Dataset/training'
trainingGtPath = './Dataset/training_gt'

valImagePath = './Dataset/val'
valGtPath = './Dataset/val_gt'

### Process COCO

In [ ]:
with open('./training3kCOCOGt.json', 'r') as file:
    train_data = json.load(file)

with open('./valtest75COCOGt.json', 'r') as file:
    val_data = json.load(file)

train_coco = {}
valtest_coco = {}

for cat, val in train_data.items():
    for imgId, seg in val.items():
        img = cocoTrain.loadImgs(ids=int(imgId))[0]
        imgLoaded = io.imread(img['coco_url'])
        if imgLoaded.ndim==2 or (imgLoaded.ndim==3 and imgLoaded.shape[2]==1):
            imgLoaded = cv2.cvtColor(imgLoaded, cv2.COLOR_GRAY2BGR)
        cv2.imwrite(os.path.join(trainingImagePath, str(imgId)+'.png'), imgLoaded)

        width, height = img['width'], img['height']
        seg_mask = np.zeros((height, width))
        for segCat, annIds in seg.items():
            segId = class_labels[segCat]
            for annId in annIds:
                seg_coco = cocoTrain.annToMask(cocoTrain.loadAnns(annId)[0])
                seg_mask[seg_coco.astype(bool)] = segId
        cv2.imwrite(os.path.join(trainingGtPath, str(imgId)+'.png'), seg_mask)
        train_coco[imgId] = {'image': os.path.join('/Dataset/training', str(imgId)+'.png'), 'segmentation': os.path.join('/Dataset/training_gt', str(imgId)+'.png')}

for cat, val in val_data.items():
    for imgId, seg in val.items():
        img = cocoVal.loadImgs(ids=int(imgId))[0]
        imgLoaded = io.imread(img['coco_url'])
        if imgLoaded.ndim==2 or (imgLoaded.ndim==3 and imgLoaded.shape[2]==1):
            imgLoaded = cv2.cvtColor(imgLoaded, cv2.COLOR_GRAY2BGR)
        cv2.imwrite(os.path.join(valImagePath, str(imgId)+'.png'), imgLoaded)

        width, height = img['width'], img['height']
        seg_mask = np.zeros((height, width))
        for segCat, annIds in seg.items():
            segId = class_labels[segCat]
            for annId in annIds:
                seg_coco = cocoVal.annToMask(cocoVal.loadAnns(annId)[0])
                seg_mask[seg_coco.astype(bool)] = segId
        cv2.imwrite(os.path.join(valGtPath, str(imgId)+'.png'), seg_mask)
        valtest_coco[imgId] = {'image': os.path.join('/Dataset/val', str(imgId)+'.png'), 'segmentation': os.path.join('/Dataset/val_gt', str(imgId)+'.png')}

### Process City

In [ ]:
training_image_store = './leftImg8bit_trainvaltest/leftImg8bit/train'
valtest_image_store = './leftImg8bit_trainvaltest/leftImg8bit/val'

In [ ]:
train_city = {}
val_city = {}

with open('./trainingCityGt.json', 'r') as file:
    train_data = json.load(file)

with open('./valtest75CityGt.json', 'r') as file:
    val_data = json.load(file)

for cat, val in train_data.items():
    for imgId, seg in val.items():
        location = imgId.split('_')[0]
        imagePath = os.path.join(training_image_store, location, imgId + '_leftImg8bit.png')
        image = Image.open(imagePath).convert('RGB')
        width, height = image.size
        image.save(os.path.join(trainingImagePath, str(imgId)+'.png'))

        seg_mask = np.zeros((height, width))
        for segCat, polygons in seg.items():
            segId = class_labels[segCat]
            for polygon in polygons:
                polygon_array = np.array(polygon, dtype=np.int32)
                seg_city = np.zeros((height, width), dtype=np.int8)
                cv2.fillPoly(seg_city, [polygon_array], 1)
                seg_mask[seg_city.astype(bool)] = segId
        cv2.imwrite(os.path.join(trainingGtPath, str(imgId)+'.png'), seg_mask)
        train_city[imgId] = {'image': os.path.join('/Dataset/training', str(imgId)+'.png'), 'segmentation': os.path.join('/Dataset/training_gt', str(imgId)+'.png')}


for cat, val in val_data.items():
    for imgId, seg in val.items():
        location = imgId.split('_')[0]
        imagePath = os.path.join(valtest_image_store, location, imgId + '_leftImg8bit.png')
        image = Image.open(imagePath).convert('RGB')
        width, height = image.size
        image.save(os.path.join(valImagePath, str(imgId)+'.png'))

        seg_mask = np.zeros((height, width))
        for segCat, polygons in seg.items():
            segId = class_labels[segCat]
            for polygon in polygons:
                polygon_array = np.array(polygon, dtype=np.int32)
                seg_city = np.zeros((height, width), dtype=np.int8)
                cv2.fillPoly(seg_city, [polygon_array], 1)
                seg_mask[seg_city.astype(bool)] = segId
        cv2.imwrite(os.path.join(valGtPath, str(imgId)+'.png'), seg_mask)
        val_city[imgId] = {'image': os.path.join('/Dataset/val', str(imgId)+'.png'), 'segmentation': os.path.join('/Dataset/val_gt', str(imgId)+'.png')}

In [ ]:
train_data = []
train_gt = []

val_temp_data = []
val_temp_gt = []

val_data = []
val_gt = []

test_data = []
test_gt = []

for imgId, detail in train_coco.items():
    train_data.append(detail['image'])
    train_gt.append(detail['segmentation'])

for imgId, detail in train_city.items():
    train_data.append(detail['image'])
    train_gt.append(detail['segmentation'])

for imgId, detail in valtest_coco.items():
    val_temp_data.append(detail['image'])
    val_temp_gt.append(detail['segmentation'])

for imgId, detail in val_city.items():
    val_temp_data.append(detail['image'])
    val_temp_gt.append(detail['segmentation'])


val_data, test_data = val_temp_data[:len(val_temp_data)//2], val_temp_data[len(val_temp_data)//2:]
val_gt, test_gt = val_temp_gt[:len(val_temp_gt)//2], val_temp_gt[len(val_temp_gt)//2:]

print(len(train_data), len(train_gt))
print(len(val_data), len(val_gt))
print(len(test_data), len(test_gt))

final_dataset = {
    'train_image': train_data,
    'train_gt': train_gt,
    'val_image': val_data,
    'val_gt': val_gt,
    'test_image': test_data,
    'test_gt': test_gt
}
with open('./dataset3k.json', 'w') as file:
    json.dump(final_dataset, file)

### Instance Count

In [ ]:
with open('./training3kCOCOGt.json', 'r') as file:
    train_data = json.load(file)

instanceCount = {}

for cat, details in train_data.items():
    for imageId, more_details in details.items():
        for innerCat, instances in more_details.items():
            if innerCat not in instanceCount:
                instanceCount[innerCat] = 0
            instanceCount[innerCat]+=len(instances)

with open('./trainingCityGt.json', 'r') as file:
    train_data = json.load(file)

for cat, details in train_data.items():
    for imageId, more_details in details.items():
        for innerCat, instances in more_details.items():
            if innerCat not in instanceCount:
                instanceCount[innerCat] = 0
            instanceCount[innerCat]+=len(instances)

print(instanceCount)

In [ ]:
with open('./dataset3k.json', 'r') as file:
    data = json.load(file)

train_data_pre = data['train_image']
train_gt_pre = data['train_gt']

val_data_pre = data['val_image']
val_gt_pre = data['val_gt']

test_data_pre = data['test_image']
test_gt_pre = data['test_gt']

In [ ]:
train_data = [os.path.join("./", curPath[1:]) for curPath in train_data_pre]
train_gt = [os.path.join("./", curPath[1:]) for curPath in train_gt_pre]

val_data = [os.path.join("./", curPath[1:]) for curPath in val_data_pre]
val_gt = [os.path.join("./", curPath[1:]) for curPath in val_gt_pre]

test_data = [os.path.join(".t/", curPath[1:]) for curPath in test_data_pre]
test_gt = [os.path.join("/./", curPath[1:]) for curPath in test_gt_pre]

In [ ]:
val_indices = [d.split('/')[-1].split('.')[0] for d in val_data]
test_indices = [d.split('/')[-1].split('.')[0] for d in test_data]

with open('./valtest75COCOGt.json', 'r') as file:
    train_data = json.load(file)

instanceCountVal = {}
instanceCountTest = {}

for cat, details in train_data.items():
    for imageId, more_details in details.items():
        if imageId in val_indices:
          for innerCat, instances in more_details.items():
              if innerCat not in instanceCountVal:
                  instanceCountVal[innerCat] = 0
              instanceCountVal[innerCat]+=len(instances)
        elif imageId in test_indices:
          for innerCat, instances in more_details.items():
              if innerCat not in instanceCountTest:
                  instanceCountTest[innerCat] = 0
              instanceCountTest[innerCat]+=len(instances)
        else:
          print('CHECK')


with open('/content/drive/MyDrive/valtest75CityGt.json', 'r') as file:
    train_data = json.load(file)

for cat, details in train_data.items():
    for imageId, more_details in details.items():
      if imageId in val_indices:
        for innerCat, instances in more_details.items():
            if innerCat not in instanceCountVal:
                instanceCountVal[innerCat] = 0
            instanceCountVal[innerCat]+=len(instances)
      elif imageId in test_indices:
            if innerCat not in instanceCountTest:
                instanceCountTest[innerCat] = 0
            instanceCountTest[innerCat]+=len(instances)
      else:
        print('CHECK')

print(instanceCountVal)
print(instanceCountTest)

### Generate CE Loss Weights

In [ ]:
train_data = [os.path.join("./", curPath[1:]) for curPath in train_data_pre]
train_gt = [os.path.join("./", curPath[1:]) for curPath in train_gt_pre]

val_data = [os.path.join("./", curPath[1:]) for curPath in val_data_pre]
val_gt = [os.path.join("./", curPath[1:]) for curPath in val_gt_pre]

test_data = [os.path.join(".t/", curPath[1:]) for curPath in test_data_pre]
test_gt = [os.path.join("/./", curPath[1:]) for curPath in test_gt_pre]

In [ ]:
# Following albumentations documentation: https://albumentations.ai/docs/3-basic-usage/semantic-segmentation/
class CocoCityDataset(Dataset):
    def __init__(self, image, seg, transform):
        self.image = image
        self.seg = seg
        self.transform = transform
    
    def __len__(self):
        return len(self.image)
    
    def __getitem__(self, index):
        image = self.image[index]
        seg = self.seg[index]

        img = cv2.cvtColor(cv2.imread(image), cv2.COLOR_BGR2RGB)
        mask = cv2.imread(seg, cv2.IMREAD_GRAYSCALE)

        if self.transform:
            transformedObjects = self.transform(image=img, mask=mask)
            img = transformedObjects['image']
            mask = transformedObjects['mask']

        return img, mask

In [ ]:
img_width, img_height = 512, 512

training_transform = A.Compose([
    A.Resize(img_width, img_height),
    A.OneOf([
        A.HorizontalFlip(),
        A.RandomRotate90(),
    ], p=0.8),
    A.GaussNoise(std_range=(0.1, 0.2), p=0.6),
    A.RandomBrightnessContrast(p=0.6),
    A.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    A.ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(img_width, img_height),
    A.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    A.ToTensorV2()
])

In [ ]:
training_dataset = CocoCityDataset(train_data, train_gt, training_transform)
training_loader = DataLoader(training_dataset, batch_size=16, shuffle=True, num_workers=4, pin_memory=True, persistent_workers=True, prefetch_factor=2)

val_dataset = CocoCityDataset(val_data, val_gt, val_transform)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True, prefetch_factor=2)

test_dataset = CocoCityDataset(test_data, test_gt, val_transform)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True, prefetch_factor=2)

In [ ]:
track_pixels = torch.zeros(6)

for batch in training_loader:
    image, mask = batch
    for classes in range(6):
      track_pixels[classes] += (mask==classes).sum()

inverse_track = 1/track_pixels
normalised_weights = inverse_track/inverse_track.sum()
print(normalised_weights)

### Run Model on Test Set

In [ ]:
class_labels = {
    'car': 1,
    'bus': 2,
    'motorcycle': 3,
    'truck': 4,
    'person': 5
}

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device used:', device)

In [ ]:
model_unet = smp.UnetPlusPlus(
    encoder_name="resnet50",
    encoder_weights="imagenet",
    in_channels=3,
    classes=6
).to(device)

state_unet = torch.load('./UNet++_7_27.pth', map_location=device)
model_unet.load_state_dict(state_unet)

In [ ]:
model_dpt = smp.DPT(
    encoder_name="tu-vit_base_patch16_384.augreg_in1k",
    encoder_weights="imagenet",
    in_channels=3,
    classes=6,
    dynamic_img_size=True
).to(device)

state_dpt = torch.load('./DPT_5_22.pth', map_location=device)
model_dpt.load_state_dict(state_dpt)

In [ ]:
model_dpt.eval()
epoch_loss = 0
result_store = []
gt_store = []

with torch.no_grad():
    for i, batch_test in enumerate(tqdm(test_loader)):
        image_test, mask_test = batch_test
        image_test = image_test.to(device)
        mask_test = mask_test.to(device)
        
        mask_test = mask_test.unsqueeze(1)

        with autocast():
            results_test = model_dpt(image_test)

        result_store.append(torch.argmax(results_test, dim=1).detach().cpu().long())
        gt_store.append(mask_test.squeeze(1).detach().cpu().long())

final_results_test = torch.cat(result_store, dim=0)
final_gt_test = torch.cat(gt_store, dim=0)

dice_score = DiceScore(num_classes=6, average='none', include_background=False, input_format='index', aggregation_level='global')
calculated_dice = dice_score(final_results_test, final_gt_test)
print('Test Dice Score:', calculated_dice)

iou_score = MeanIoU(num_classes=6, include_background=False, per_class=True, input_format='index')
calculated_iou = iou_score(final_results_test, final_gt_test)
print('Test IOU Score:', calculated_iou)

mean_metric = 0.5*torch.nanmean(calculated_dice) + 0.5*torch.mean(calculated_iou[calculated_iou>=0])
print('Test combined score:', mean_metric)

### Visualise Segmentations

In [ ]:
img_width, img_height = 512, 512

test_dataset_single = CocoCityDataset(test_data, test_gt, val_transform)
test_loader_single = DataLoader(test_dataset_single, batch_size=1, shuffle=False, num_workers=1)

In [ ]:
model_unet.eval()
result_store = []

with torch.no_grad():
    for i, batch_test in enumerate(tqdm(test_loader_single)):
        image_test, mask_test = batch_test
        image_test = image_test.to(device)
        mask_test = mask_test.to(device)
        
        mask_test = mask_test.unsqueeze(1)

        with autocast():
            results_test = model_unet(image_test)

        img = cv2.cvtColor(cv2.imread(test_data[i]), cv2.COLOR_BGR2RGB)
        img_h, img_w = img.shape[:2]

        result_test_batch = torch.argmax(results_test, dim=1).detach().cpu().long()
        mask_test_batch = mask_test.squeeze(1).detach().cpu().long()

        dice_score = DiceScore(num_classes=6, average='none', include_background=False, input_format='index', aggregation_level='global')
        calculated_dice = dice_score(result_test_batch, mask_test_batch)

        iou_score = MeanIoU(num_classes=6, include_background=False, per_class=True, input_format='index')
        calculated_iou = iou_score(result_test_batch, mask_test_batch)

        mean_metric = 0.5*torch.nanmean(calculated_dice) + 0.5*torch.mean(calculated_iou[calculated_iou>=0])

        result_test_pre_up = torch.argmax(results_test, dim=1).unsqueeze(0)
        mask_test_pre_up = mask_test

        result_test_up = F.interpolate(result_test_pre_up.float(), size=(img_h, img_w), mode='nearest')
        mask_test_up = F.interpolate(mask_test_pre_up.float(), size=(img_h, img_w), mode='nearest')


        result_test_up = result_test_up.squeeze(0).squeeze(0)
        mask_test_up = mask_test_up.squeeze(0).squeeze(0)

        store = {}
        store['image_path'] = test_data[i]
        store['gt_path'] = test_gt[i]
        store['image'] = image_test
        store['result'] = result_test_up.detach().cpu()
        store['mask'] = mask_test_up.detach().cpu()
        store['dice'] = calculated_dice
        store['iou'] = calculated_iou
        store['metric'] = mean_metric

        result_store.append(store)

In [ ]:
sorted_result = sorted(result_store, key=lambda x: x['metric'], reverse=True)

In [ ]:
val = sorted_result[-5]
image_path = val['image_path']
gt_path = val['gt_path']
image_result = val['result']
image_mask = val['mask']

dice_score = val['dice']
iou_score = val['iou']
mean_metric = val['metric']

img = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)
mask_og = cv2.imread(gt_path, cv2.IMREAD_GRAYSCALE)

print('GT classes:', np.unique(mask_og))
print('Predicted classes:', np.unique(mask_og))

print(dice_score)
print(iou_score)
print(mean_metric)

img_h, img_w = img.shape[:2]

image_result = image_result.numpy().astype(np.int8)
image_mask = image_mask.numpy().astype(np.int8)

print('Predicted classes:', np.unique(image_result))

label_colour = {
    1: 'b',
    2: 'g',
    3: 'r',
    4: 'm',
    5: 'y'
}

factor = 50
figsize = (img_w/factor, img_h/factor)
plt.figure(figsize=figsize, dpi=factor)

plt.imshow(img)
plt.axis('off')
for val, colour_label in label_colour.items():
    mask_val = (mask_og==val)
    if np.sum(mask_val)==0:
      continue
    mask = np.ma.masked_where(mask_val==0, mask_val)
    colour_map = ListedColormap([colour_label])
    plt.imshow(mask, cmap=colour_map, alpha=0.5, vmin=0, vmax=1)

plt.show()

plt.figure(figsize=figsize, dpi=factor)

plt.imshow(img)
plt.axis('off')
for val, colour_label in label_colour.items():
    mask_val = (image_result==val)
    if np.sum(mask_val)==0:
      continue
    mask = np.ma.masked_where(mask_val==0, mask_val)
    colour_map = ListedColormap([colour_label])
    plt.imshow(mask, cmap=colour_map, alpha=0.5, vmin=0, vmax=1)

plt.show()